# ⚾ 주무기(비-패스트볼)를 장타 맞았을 때 회피하는 선택의 가치

* **작성자:** DY
* **작성일:** 2026-10-05

## 1. 배경: 노트북 09와 무엇이 다른가

노트북 09는 "장타를 맞은 구종을 재대결에서 피하는 게 좋은 선택인가"를 **선택가치**(GBM으로 "실제 선택"과
"맞은 구종 재사용"을 반사실(counterfactual)로 비교)라는 지표로 검증했습니다. 통계적으로는 유의했지만
(`avoided` +0.0023, 보수적으로도 p≈0.0016), 실전 단위로 환산하면 투수 1명이 6개 시즌 동안 얻는 이득이
1득점에도 못 미치는 수준이라 "유의하지만 중요하지 않다"는 결론이었습니다.

이 노트북은 같은 선택가치 지표를, **"맞은 구종이 그 투수·시즌의 주무기(가장 많이 던지는 비-패스트볼
구종)였던 경우**로 좁혀서 다시 봅니다. 두 가지 이유가 있습니다.

1. **반사실 예측이 더 신뢰할 만합니다.** 선택가치는 "맞은 구종을 재사용했다면"을 GBM으로 예측해야
   하는데, 노트북 09의 일반 표본에서는 이 맞은 구종이 그 투수가 거의 안 던지는 구종인 경우가 섞여
   있어 GBM이 외삽(extrapolation)해야 했습니다. **주무기로 좁히면 반사실 자체가 그 투수가 가장 많이
   던지는 구종**이 되어, GBM 학습 데이터(결정구 917,717건)에 가장 풍부하게 있는 조합을 묻는 질문이
   됩니다 — 외삽이 아니라 보간에 가까워집니다.
2. **더 의미 있는 질문입니다.** 2군 구종을 피하는 건 리스크가 적지만, 주무기를 포기하는 건 더 큰
   전략적 결정입니다. 효과가 실재한다면 여기서 더 크게 나타날 가능성이 있습니다.

패스트볼 계열은 "주무기" 후보에서 제외합니다 — 대부분 투수의 최다 사용 구종은 패스트볼이라, 그대로
두면 "주무기"가 사실상 "패스트볼"의 동의어가 되어 버려 의도한 질문(진짜 승부구를 포기하는 선택)을
묻지 못합니다.

투수당 표본 하한은 두지 않습니다 — 노트북 07·09와 같은 이유로, `(1 | pitcher)` 랜덤절편의 부분
풀링(partial pooling)이 저표본 투수를 적절히 수축시켜 반영합니다.

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

from src.analysis.avoidance_stats import compute_season_usage_rate
from src.analysis.glmer_runner import (
    check_convergence,
    extract_lmer_fixed_effects,
    extract_variance_components,
    fit_lmer,
)
from src.analysis.mixed_effects_model import compute_icc_gaussian
from src.analysis.run_same_batter_rematch_analysis import build_xbh_rematch
from src.preprocessing.pitch_family import map_pitch_family

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

TRAIN_SEASONS = (2021, 2022, 2023, 2024, 2025)
VALID_SEASON = 2026
RANDOM_STATE = 20261003

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 2. 데이터 준비와 선택가치 계산 (노트북 09와 동일)

팀 CSV 로딩부터 "타석 전체로 집계한 선택가치"(`selection_value_pa_mean`)까지는 노트북 09와 완전히
같은 과정입니다 — 반사실 구종(맞은 구종)을 그대로 쓰되, 이 노트북에서는 뒤에서 "주무기였던 경우"로만
표본을 좁힙니다.

In [2]:
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs
from src.preprocessing.build_next_ab_dataset import EXTRA_BASE_HIT_EVENTS

LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score',
    'woba_value', 'woba_denom',
]


def load_team_csv() -> pd.DataFrame:
    frames = [
        pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False)
        for path in list_research_csvs()
    ]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    return pitches


pitches = load_team_csv()
assert len(pitches) == 3_592_302 and pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명')

3,592,302행, 투수 1,067명


In [3]:
def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['score_diff'] = np.where(
        out['inning_topbot'] == 'Top', out['home_score'] - out['away_score'], out['away_score'] - out['home_score']
    )
    out['platoon_match'] = np.where(
        out['stand'].isna() | out['p_throws'].isna(), np.nan, (out['stand'] == out['p_throws']).astype(float)
    )
    return out


_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan], 'on_2b': [np.nan, np.nan, 2.0], 'on_3b': [np.nan, np.nan, np.nan],
    'inning_topbot': ['Top', 'Bot', 'Top'], 'home_score': [3, 5, 2], 'away_score': [3, 2, 4],
    'stand': ['L', 'R', 'R'], 'p_throws': ['R', 'R', 'L'],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1]
assert _r['risp'].tolist() == [0, 0, 1]
assert _r['score_diff'].tolist() == [0, -3, -2]
assert _r['platoon_match'].tolist() == [0.0, 1.0, 0.0]
print('add_situation_columns 테스트 통과')

pitches = add_situation_columns(pitches)

add_situation_columns 테스트 통과


In [4]:
season_usage = compute_season_usage_rate(pitches)
xbh = build_xbh_rematch(pitches, season_usage)
xbh = xbh[xbh['has_next_ab']].copy()
assert len(xbh) == 27_391, len(xbh)
print(f'재대결 이벤트 {len(xbh):,}건')

2026-10-05 23:29:38,048 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


재대결 이벤트 27,391건


In [5]:
MODEL_FEATURES = [
    'pitch_type', 'stand', 'p_throws', 'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff',
    'season_usage_rate',
]
CATEGORICAL_FEATURES = ['pitch_type', 'stand', 'p_throws']


def build_decisive_pitch_table(pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna() & pitches_sit['pitch_type'].notna()].copy()
    decisive = decisive.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    for col in CATEGORICAL_FEATURES:
        decisive[col] = decisive[col].astype('category')
    return decisive


decisive_table = build_decisive_pitch_table(pitches, season_usage)
assert decisive_table['season_usage_rate'].notna().mean() > 0.99
print(f'결정구 {len(decisive_table):,}건, season_usage_rate 결측 {decisive_table["season_usage_rate"].isna().sum()}건')

결정구 917,717건, season_usage_rate 결측 0건


In [6]:
def fit_run_value_model(table: pd.DataFrame, seasons) -> HistGradientBoostingRegressor:
    train = table[table['season'].isin(seasons)]
    model = HistGradientBoostingRegressor(
        categorical_features='from_dtype', random_state=RANDOM_STATE, max_iter=200, early_stopping=True,
    )
    model.fit(train[MODEL_FEATURES], train['woba_value'])
    return model


train_set = decisive_table[decisive_table['season'].isin(TRAIN_SEASONS)]
valid_set = decisive_table[decisive_table['season'] == VALID_SEASON]
model_cv = fit_run_value_model(decisive_table, TRAIN_SEASONS)
pred_valid = model_cv.predict(valid_set[MODEL_FEATURES])
model_r2 = r2_score(valid_set['woba_value'], pred_valid)
print(f'(전체 피처) GBM 검증 R²: {model_r2:.4f}')
assert model_r2 > 0

final_model = fit_run_value_model(decisive_table, decisive_table['season'].unique())
print('(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀')

(전체 피처) GBM 검증 R²: 0.0568


(전체 피처) 최종 모델 준비 완료 -- 선택가치 계산에 씀


In [7]:
def find_rematch_outcome_pitch(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna()][
        ['game_pk', 'pitcher', 'season', 'at_bat_number', *MODEL_FEATURES, 'platoon_match', 'events', 'woba_value']
    ]
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'reused_same_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    out = keys.merge(decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')
    return out


REQUIRED_OUTCOME_COLUMNS = sorted(set(['events', 'woba_value', 'platoon_match', *MODEL_FEATURES]))


def build_avoidance_outcome_dataset(found: pd.DataFrame) -> pd.DataFrame:
    out = found.copy()
    out['pitch_family'] = out['pitch_type'].map(map_pitch_family)
    out = out.dropna(subset=[*REQUIRED_OUTCOME_COLUMNS, 'pitch_family', 'reused_same_type'])
    out['avoided'] = (1 - out['reused_same_type']).astype(int)
    out['allowed_xbh_again'] = out['events'].isin(EXTRA_BASE_HIT_EVENTS).astype(int)
    out['pitcher'] = out['pitcher'].astype(str)
    return out


found = find_rematch_outcome_pitch(xbh, decisive_table)
found_with_pa = found.dropna(subset=['woba_value'])
model_data = build_avoidance_outcome_dataset(found_with_pa)
assert len(model_data) == 27_245, len(model_data)
print(f'모델링에 쓸 재대결: {len(model_data):,}건, 투수 {model_data["pitcher"].nunique()}명 (노트북 09와 동일해야 함)')

모델링에 쓸 재대결: 27,245건, 투수 695명 (노트북 09와 동일해야 함)


In [8]:
def attach_counterfactual_pitch_type(model_data_: pd.DataFrame, xbh_: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    cf_keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cf_keys['pitcher'] = cf_keys['pitcher'].astype(str)
    out = model_data_.merge(cf_keys, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')

    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    ).copy()
    usage['pitcher'] = usage['pitcher'].astype(str)
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    return out


model_data = attach_counterfactual_pitch_type(model_data, xbh, season_usage)
assert model_data['hit_pitch_type'].notna().all()
print('반사실 구종 결측 0건 확인')

반사실 구종 결측 0건 확인


In [9]:
def compute_selection_value(df: pd.DataFrame, model) -> pd.Series:
    actual_pred = model.predict(df[MODEL_FEATURES])
    cf = df[MODEL_FEATURES].copy()
    cf['pitch_type'] = df['hit_pitch_type'].astype(df['pitch_type'].dtype)
    cf['season_usage_rate'] = df['hit_pitch_season_usage_rate']
    cf_pred = model.predict(cf)
    return pd.Series(cf_pred - actual_pred, index=df.index)


class _LinearStub:
    def predict(self, X):
        return np.where(X['pitch_type'].astype(str) == 'SL', 0.5, 0.2)


_cat_dtype = pd.CategoricalDtype(categories=['SL', 'FF', 'CH'])
_df_sv = pd.DataFrame({
    'pitch_type': pd.Categorical(['FF', 'SL'], dtype=_cat_dtype), 'hit_pitch_type': ['SL', 'SL'],
    'season_usage_rate': [0.4, 0.5], 'hit_pitch_season_usage_rate': [0.3, 0.5],
    'stand': ['R', 'R'], 'p_throws': ['R', 'R'], 'balls': [0, 0], 'strikes': [0, 0],
    'outs_when_up': [0, 0], 'runners_n': [0, 0], 'risp': [0, 0], 'score_diff': [0, 0],
})
_sv = compute_selection_value(_df_sv, _LinearStub())
assert np.allclose(_sv.to_numpy(), [0.3, 0.0])
print('compute_selection_value 테스트 통과')

compute_selection_value 테스트 통과


In [10]:
def build_rematch_pa_pitches(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame, pitch_type_dtype) -> pd.DataFrame:
    keys = xbh_[['game_pk', 'pitcher', 'season', 'next_at_bat_number', 'hit_pitch_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    cols = ['game_pk', 'pitcher', 'at_bat_number', 'pitch_type', 'stand', 'p_throws',
            'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff']
    out = keys.merge(pitches_sit[cols], on=['game_pk', 'pitcher', 'at_bat_number'], how='left')
    out = out.dropna(subset=['pitch_type'])

    out = out.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    usage = season_usage_.rename(
        columns={'pitch_type': 'hit_pitch_type', 'season_usage_rate': 'hit_pitch_season_usage_rate'}
    )
    out = out.merge(
        usage[['pitcher', 'season', 'hit_pitch_type', 'hit_pitch_season_usage_rate']],
        on=['pitcher', 'season', 'hit_pitch_type'], how='left',
    )
    out['hit_pitch_season_usage_rate'] = out['hit_pitch_season_usage_rate'].fillna(0.0)
    out = out.dropna(subset=['season_usage_rate'])
    out['pitch_type'] = out['pitch_type'].astype(pitch_type_dtype)
    out['pitcher'] = out['pitcher'].astype(str)
    for col in ('stand', 'p_throws'):
        out[col] = out[col].astype('category')
    return out


def aggregate_pa_selection_value(pa_pitches: pd.DataFrame, model) -> pd.Series:
    sv = compute_selection_value(pa_pitches, model)
    return pa_pitches.assign(_sv=sv).groupby(['game_pk', 'pitcher', 'at_bat_number'])['_sv'].mean()


_TRAINED_DTYPE = pd.CategoricalDtype(categories=['SL', 'FF', 'CU', 'CH'])
_xbh_pa = pd.DataFrame({
    'game_pk': [100], 'pitcher': [10], 'season': [2021], 'next_at_bat_number': [5.0], 'hit_pitch_type': ['SL'],
})
_px_pa = pd.DataFrame({
    'game_pk': [100, 100, 100, 100], 'pitcher': [10, 10, 10, 10], 'at_bat_number': [5, 5, 5, 9],
    'pitch_type': ['FF', 'CU', 'FF', 'SL'], 'stand': ['R'] * 4, 'p_throws': ['R'] * 4,
    'balls': [0, 1, 2, 0], 'strikes': [0, 1, 2, 0], 'outs_when_up': [0, 0, 0, 0],
    'runners_n': [0, 0, 0, 0], 'risp': [0, 0, 0, 0], 'score_diff': [0, 0, 0, 0],
})
_su_pa = pd.DataFrame({
    'pitcher': [10, 10, 10], 'season': [2021, 2021, 2021], 'pitch_type': ['FF', 'CU', 'SL'],
    'season_usage_rate': [0.5, 0.3, 0.2],
})
_pa_pitches = build_rematch_pa_pitches(_xbh_pa, _px_pa, _su_pa, _TRAINED_DTYPE)
assert len(_pa_pitches) == 3
assert _pa_pitches['pitch_type'].astype(str).tolist() == ['FF', 'CU', 'FF']
print('build_rematch_pa_pitches 테스트 통과')

_pa_full_avoid = pd.DataFrame({
    'game_pk': [100] * 3, 'pitcher': ['10'] * 3, 'at_bat_number': [5] * 3,
    'pitch_type': pd.Categorical(['FF', 'CU', 'FF'], dtype=_TRAINED_DTYPE), 'hit_pitch_type': ['SL'] * 3,
    'season_usage_rate': [0.5, 0.3, 0.5], 'hit_pitch_season_usage_rate': [0.2, 0.2, 0.2],
    'stand': ['R'] * 3, 'p_throws': ['R'] * 3, 'balls': [0, 1, 2], 'strikes': [0, 1, 2],
    'outs_when_up': [0, 0, 0], 'runners_n': [0, 0, 0], 'risp': [0, 0, 0], 'score_diff': [0, 0, 0],
})
_out_a = aggregate_pa_selection_value(_pa_full_avoid, _LinearStub())
assert np.isclose(_out_a.loc[(100, '10', 5)], 0.3)
print('aggregate_pa_selection_value 테스트 통과')

build_rematch_pa_pitches 테스트 통과
aggregate_pa_selection_value 테스트 통과


In [11]:
pa_pitches = build_rematch_pa_pitches(xbh, pitches, season_usage, decisive_table['pitch_type'].dtype)
pa_mean = aggregate_pa_selection_value(pa_pitches, final_model).rename('selection_value_pa_mean').reset_index()
model_data = model_data.merge(pa_mean, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='one_to_one')
assert model_data['selection_value_pa_mean'].notna().all()
print(f'노트북 09와 동일한 전체 표본 준비 완료: {len(model_data):,}건, 투수 {model_data["pitcher"].nunique()}명')

노트북 09와 동일한 전체 표본 준비 완료: 27,245건, 투수 695명


## 3. 주무기(비-패스트볼) 필터링

"맞은 구종"이 그 투수·시즌의 `season_usage_rate` 1위 구종(패스트볼 계열 제외)이었던 재대결만 남깁니다.

In [12]:
def filter_to_primary_weapon_hits(model_data_: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    su = season_usage_.copy()
    su['pitch_family'] = su['pitch_type'].map(map_pitch_family)
    su = su[su['pitch_family'] != 'fastball']
    primary = su.sort_values('season_usage_rate', ascending=False).drop_duplicates(['pitcher', 'season'])
    primary = primary[['pitcher', 'season', 'pitch_type']].rename(columns={'pitch_type': 'primary_pitch_type'})
    primary['pitcher'] = primary['pitcher'].astype(str)

    out = model_data_.merge(primary, on=['pitcher', 'season'], how='left')
    out = out[out['hit_pitch_type'] == out['primary_pitch_type']].drop(columns='primary_pitch_type')
    return out

In [13]:
_su_fix = pd.DataFrame({
    'pitcher': [10, 10, 10, 20, 20], 'season': [2021, 2021, 2021, 2021, 2021],
    'pitch_type': ['FF', 'SL', 'CH', 'FF', 'CU'], 'season_usage_rate': [0.5, 0.3, 0.2, 0.6, 0.4],
})
_md_fix = pd.DataFrame({
    'pitcher': ['10', '10', '20'], 'season': [2021, 2021, 2021],
    'hit_pitch_type': ['SL', 'CH', 'CU'], 'avoided': [1, 0, 1],
})
_filtered = filter_to_primary_weapon_hits(_md_fix, _su_fix)
assert _filtered['hit_pitch_type'].tolist() == ['SL', 'CU']  # pitcher 10's CH row dropped -- not their non-FF primary

_su_fix2 = pd.DataFrame({'pitcher': [30], 'season': [2021], 'pitch_type': ['FF'], 'season_usage_rate': [1.0]})
_md_fix2 = pd.DataFrame({'pitcher': ['30'], 'season': [2021], 'hit_pitch_type': ['FF'], 'avoided': [0]})
assert len(filter_to_primary_weapon_hits(_md_fix2, _su_fix2)) == 0  # no non-fastball pitch -> dropped
print('filter_to_primary_weapon_hits 테스트 통과')

filter_to_primary_weapon_hits 테스트 통과


In [14]:
primary_data = filter_to_primary_weapon_hits(model_data, season_usage)
print(f'주무기(비-패스트볼)를 맞은 재대결: {len(primary_data):,}건 ({len(primary_data) / len(model_data):.1%}), '
      f'투수 {primary_data["pitcher"].nunique()}명')
display(primary_data.groupby('avoided')['selection_value_pa_mean'].describe().round(4))

주무기(비-패스트볼)를 맞은 재대결: 6,083건 (22.3%), 투수 576명


,count,mean,std,min,25%,50%,75%,max
avoided,,,,,,,,
0,2415.0,-0.0085,0.0147,-0.1685,-0.0169,-0.0072,0.0002,0.0696
1,3668.0,-0.0077,0.0237,-0.3029,-0.0212,-0.0061,0.0079,0.1121


## 4. 혼합효과 모델 적합 (주무기 표본)

노트북 09와 같은 공식을, 주무기 표본(`primary_data`)에 적합합니다.

In [15]:
FORMULA_SELECTION = (
    'selection_value_pa_mean ~ avoided + outs_when_up + score_diff + platoon_match + risp + runners_n '
    '+ factor(season) + (1 | pitcher)'
)
fit_lmer(primary_data, FORMULA_SELECTION)
print('=== 수렴 상태 ===')
print(check_convergence())
fixed_primary = extract_lmer_fixed_effects()
print('=== 고정효과 (avoided 포함) ===')
display(fixed_primary.round(4))
vc_primary = extract_variance_components('pitcher')
print('분산 성분:', vc_primary)
if vc_primary['pitcher_intercept'] is None or vc_primary['residual'] is None:
    icc_primary = None
    print('ICC 계산 불가')
else:
    icc_primary = compute_icc_gaussian(vc_primary['pitcher_intercept'], vc_primary['residual'])
    print(f'ICC: {icc_primary:.4f}')

2026-10-05 23:30:06,482 [WARNING] R callback write-console: Loading required package: Matrix
  


=== 수렴 상태 ===
OK: no convergence warnings; isSingular=False
=== 고정효과 (avoided 포함) ===


,term,estimate,std_error,t_value,ci_low,ci_high,p_value
0,(Intercept),-0.0086,0.0009,-9.1572,-0.0105,-0.0068,0.0000
1,avoided,0.0003,0.0005,0.7210,-0.0006,0.0012,0.4709
2,outs_when_up,0.0001,0.0003,0.5121,-0.0004,0.0007,0.6086
3,score_diff,0.0001,0.0001,1.0422,-0.0001,0.0002,0.2973
4,platoon_match,0.0018,0.0005,3.6430,0.0008,0.0027,0.0003
5,risp,-0.0003,0.0008,-0.3026,-0.0019,0.0014,0.7622
6,runners_n,0.0021,0.0005,4.3759,0.0011,0.0030,0.0000
7,factor(season)2022,-0.0011,0.0008,-1.3224,-0.0027,0.0005,0.1860
8,factor(season)2023,-0.0022,0.0008,-2.6902,-0.0038,-0.0006,0.0071
9,factor(season)2024,-0.0011,0.0009,-1.2501,-0.0028,0.0006,0.2113


분산 성분: {'pitcher_intercept': 0.00014216417113815068, 'residual': 0.0002870943340290706}
ICC: 0.3312


## 5. 견고성 검증: GBM 재적합 불확실성

노트북 09의 (C) 검증과 같은 방법입니다. 이번 표본은 더 작아서(6,119건대 27,245건) 1단계 불확실성의
상대적 비중이 더 클 수 있습니다.

In [16]:
def gbm_refit_bootstrap_gap_sd(decisive_table_, pa_pitches_, avoided_by_pa, seasons, n_boot, random_state):
    rng = np.random.default_rng(random_state)
    train = decisive_table_[decisive_table_['season'].isin(seasons)]
    gaps = []
    for b in range(n_boot):
        resample_idx = rng.integers(0, len(train), size=len(train))
        boot_train = train.iloc[resample_idx]
        model = HistGradientBoostingRegressor(
            categorical_features='from_dtype', random_state=random_state + b, max_iter=200, early_stopping=True,
        )
        model.fit(boot_train[MODEL_FEATURES], boot_train['woba_value'])
        pa_mean = aggregate_pa_selection_value(pa_pitches_, model).reindex(avoided_by_pa.index)
        gaps.append(float(pa_mean[avoided_by_pa == 1].mean() - pa_mean[avoided_by_pa == 0].mean()))
    return gaps


_avoided_by_pa_primary = primary_data.set_index(['game_pk', 'pitcher', 'at_bat_number'])['avoided']
_pa_pitches_primary = pa_pitches.merge(
    primary_data[['game_pk', 'pitcher', 'at_bat_number']], on=['game_pk', 'pitcher', 'at_bat_number'],
)
N_BOOT = 50
boot_gaps = gbm_refit_bootstrap_gap_sd(
    decisive_table, _pa_pitches_primary, _avoided_by_pa_primary, decisive_table['season'].unique(), N_BOOT, RANDOM_STATE,
)
boot_sd = float(np.std(boot_gaps))
_lmer_se = float(fixed_primary.loc[fixed_primary['term'] == 'avoided', 'std_error'].iloc[0])
_combined_se = float(np.sqrt(_lmer_se ** 2 + boot_sd ** 2))
print(f'GBM 재적합 부트스트랩 {N_BOOT}회: avoided 효과(평균 차이)의 표준편차 = {boot_sd:.5f}')
print(f'lmer가 보고한 SE = {_lmer_se:.5f}')
print(f'두 불확실성을 합친(제곱합 제곱근) 보수적 SE = {_combined_se:.5f}')

GBM 재적합 부트스트랩 50회: avoided 효과(평균 차이)의 표준편차 = 0.00126
lmer가 보고한 SE = 0.00047
두 불확실성을 합친(제곱합 제곱근) 보수적 SE = 0.00134


## 6. 견고성 검증: 순열검정

노트북 09의 (D) 검증과 같은 방법입니다.

In [17]:
def permutation_test_avoided_effect(values, avoided, n_perm, seed):
    rng = np.random.default_rng(seed)
    avoided = np.asarray(avoided)
    values = np.asarray(values, dtype=float)
    observed = float(values[avoided == 1].mean() - values[avoided == 0].mean())
    n = len(values)
    n1 = int(avoided.sum())
    null_gaps = np.empty(n_perm)
    for i in range(n_perm):
        perm_idx = rng.permutation(n)
        chosen, rest = perm_idx[:n1], perm_idx[n1:]
        null_gaps[i] = values[chosen].mean() - values[rest].mean()
    p_value = float((np.abs(null_gaps) >= abs(observed)).mean())
    return {'observed_gap': observed, 'p_value': p_value, 'null_sd': float(null_gaps.std())}


_v = np.array([0.0] * 5 + [1.0] * 5)
_a = np.array([0] * 5 + [1] * 5)
_res = permutation_test_avoided_effect(_v, _a, n_perm=500, seed=1)
assert np.isclose(_res['observed_gap'], 1.0)
assert _res['p_value'] < 0.05
print('permutation_test_avoided_effect 테스트 통과')

permutation_test_avoided_effect 테스트 통과


In [18]:
_perm_result = permutation_test_avoided_effect(
    primary_data['selection_value_pa_mean'].to_numpy(), primary_data['avoided'].to_numpy(),
    n_perm=5000, seed=RANDOM_STATE,
)
print(f'관측된 평균 차이: {_perm_result["observed_gap"]:.5f}')
print(f'순열검정 p-value (5000회): {_perm_result["p_value"]:.5f}')
print(f'귀무분포 표준편차: {_perm_result["null_sd"]:.5f}')

관측된 평균 차이: 0.00081
순열검정 p-value (5000회): 0.14220
귀무분포 표준편차: 0.00054


## 7. 실전 단위 환산과 노트북 09와의 비교

In [19]:
WOBA_SCALE = 1.2

selection_effect_woba = float(fixed_primary.loc[fixed_primary['term'] == 'avoided', 'estimate'].iloc[0])
runs_per_decision = selection_effect_woba / WOBA_SCALE
rematches_per_pitcher_primary = float(primary_data.groupby('pitcher').size().mean())
n_seasons_in_sample = int(pitches['season'].nunique())
total_runs_over_sample = runs_per_decision * rematches_per_pitcher_primary
runs_per_pitcher_per_season = total_runs_over_sample / n_seasons_in_sample

print(f'주무기 표본 avoided 효과: {selection_effect_woba:.4f} wOBA ≈ {runs_per_decision:.5f} 득점/재대결')
print(f'주무기를 맞은 투수 1명당 평균 재대결 수: {rematches_per_pitcher_primary:.1f}건')
print(f'→ {n_seasons_in_sample}개 시즌 누적 총 이득: 약 {total_runs_over_sample:.3f}득점, '
      f'시즌당 약 {runs_per_pitcher_per_season:.4f}득점')
print('\n=== 노트북 09(전체 표본) vs 노트북 10(주무기 표본) 비교 ===')
print(f'전체 표본: avoided +0.0023 (p≈0.0016 보수적 기준), n=27,245')
print(f'주무기 표본: avoided {selection_effect_woba:+.4f} (n={len(primary_data):,})')

주무기 표본 avoided 효과: 0.0003 wOBA ≈ 0.00028 득점/재대결
주무기를 맞은 투수 1명당 평균 재대결 수: 10.6건
→ 6개 시즌 누적 총 이득: 약 0.003득점, 시즌당 약 0.0005득점

=== 노트북 09(전체 표본) vs 노트북 10(주무기 표본) 비교 ===
전체 표본: avoided +0.0023 (p≈0.0016 보수적 기준), n=27,245
주무기 표본: avoided +0.0003 (n=6,083)


## 8. 해석과 한계

* **표본.** 전체 재대결 27,245건 중 맞은 구종이 그 시즌 비-패스트볼 주무기였던 경우는 **6,083건
  (22.3%), 투수 576명**입니다.
* **결과 — 효과가 사라집니다.** `avoided` 추정치는 **+0.0003** [95% CI −0.0006, 0.0012], **p=0.4709**로
  유의하지 않습니다. 5,000회 순열검정도 p=0.1422로 같은 결론입니다. 노트북 09의 전체 표본(+0.0023,
  p<0.0001)과 비교하면 크기는 약 1/8, 유의성은 완전히 사라졌습니다 — **더 크게 나타날 거라는 가설과
  반대 방향의 결과**입니다.
* **왜일까 — 기준선 자체가 뒤집혀 있습니다.** `avoided`별 `selection_value_pa_mean` 평균이 두 그룹
  다 **음수**입니다(avoided=0: −0.0085, avoided=1: −0.0077). 노트북 09의 전체 표본에서는 두 그룹 다
  양수였던 것과 반대입니다. 이건 "주무기를 재사용했다고 가정"하는 반사실 예측 자체가, GBM 기준으로는
  평균적으로 **실제로 한 선택보다 더 낮은 기대 피해**로 나온다는 뜻입니다 — 즉 GBM은 "이 투수의
  `season_usage_rate`가 가장 높은 구종(주무기)은 대체로 안전하다"는 패턴을 학습했고, 이게 `avoided`
  여부와 무관하게 두 그룹 모두를 음수 쪽으로 끌어내립니다. `avoided`의 효과(두 그룹 간 차이)는
  그 위에서 미미하게만 남습니다.
* **ICC가 크게 뛰었습니다 — 0.0029 → 0.3312.** 노트북 09(전체 표본)에서는 투수 간 분산 비율이
  0.3%에 불과했는데, 이 주무기 표본에서는 **33%**로 뛰었습니다. "그 투수가 누구인지"가 이 좁은 표본
  에서는 선택가치의 상당 부분을 설명한다는 뜻이고, 이는 투수마다 주무기의 질 자체가 다르다는 사실을
  반영할 가능성이 큽니다 — `avoided`라는 그 타석의 선택보다, "이 투수의 주무기가 원래 얼마나 좋은가"가
  더 큰 변수가 된 것으로 보입니다.
* **견고성.** GBM 재적합 불확실성(SD=0.00126)은 `lmer`의 SE(0.00047)보다 **2.7배** 커서, 노트북
  09(2.3배)보다 1단계 불확실성의 상대적 비중이 더 큽니다 — 표본이 작아지면서 이 문제가 더 심해진다는
  예상과 일치합니다.
* **실전 단위.** 7절 환산대로 투수 1명당 6개 시즌 누적 약 0.003득점(시즌당 0.0005득점)으로, 노트북
  09의 이미 작았던 수치(0.074득점/6시즌)보다도 한 자릿수 더 작습니다.
* **종합.** "반사실 예측을 더 신뢰할 만하게 만들면 효과가 더 크게 나타날 것"이라는 가설은 **기각**
  됐습니다. 오히려 노트북 09의 전체 표본에서 관측된 유의한 효과가, 이 더 깨끗한 부분표본에서는 거의
  전부 투수 간 차이(ICC 급등)로 흡수되고 `avoided` 자체의 몫은 거의 남지 않았습니다. 이는 노트북
  09의 발견이 "회피라는 선택 자체"보다 "애초에 좋은 주무기를 가진 투수인가"와 더 관련 있을 수
  있다는, 선택편향 쪽에 힘을 싣는 결과입니다.
* **외부 타당성(external validity)의 축소.** 이 결과는 "주무기(비-패스트볼)를 장타 맞은 경우"에만
  적용됩니다 — 일반화할 수 없습니다.
* **노트북 09에서 넘어온, 여전히 남은 한계.** 선택편향(그날 컨디션 등 관측되지 않는 공통원인,
  `pitcher_id`를 뺀 설계상 해결 불가능 — 이번 노트북의 ICC 급등이 오히려 이 문제의 실재를 뒷받침),
  다중비교 미보정(노트북 09·10 두 분석 모두 포함하면 비교 대상이 더 늘어남), `season_usage_rate`의
  시즌 전체 집계가 미래 정보를 포함한다는 점은 그대로 적용됩니다.